In [1]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression as SklearnLR
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print('All imports loaded successfully!')


All imports loaded successfully!


### Load and explore Telco Customer Churn Dataset

In [2]:
# Load dataset
df_raw = pd.read_csv('../data/Telco-Customer-Churn.csv')

print(f'Dataset shape: {df_raw.shape}')
print(f'Columns: {df_raw.shape[1]}')
for i, col in enumerate(df_raw.columns):
    print(f'  {i+1:2d}. {col} ({df_raw[col].dtype})')
print(f'\nFirst 3 rows:')
df_raw.head(3)

Dataset shape: (7043, 21)
Columns: 21
   1. customerID (str)
   2. gender (str)
   3. SeniorCitizen (int64)
   4. Partner (str)
   5. Dependents (str)
   6. tenure (int64)
   7. PhoneService (str)
   8. MultipleLines (str)
   9. InternetService (str)
  10. OnlineSecurity (str)
  11. OnlineBackup (str)
  12. DeviceProtection (str)
  13. TechSupport (str)
  14. StreamingTV (str)
  15. StreamingMovies (str)
  16. Contract (str)
  17. PaperlessBilling (str)
  18. PaymentMethod (str)
  19. MonthlyCharges (float64)
  20. TotalCharges (str)
  21. Churn (str)

First 3 rows:


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [7]:
# Dataset overview — types, nulls, hidden issues
print('='*65)
print('DATASET OVERVIEW')
print('='*65)

print(f'\nShape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
print(f'\nTarget distribution:')
print(df_raw['Churn'].value_counts())
print(f'Churn rate: {(df_raw["Churn"] == "Yes").mean():.1%}')

print(f'\n--- Column Types ---')
print(df_raw.dtypes.value_counts())

print(f'\n--- Missing Values ---')
print(f'Explicit NaN count: {df_raw.isnull().sum().sum()}')

# TotalCharges has hidden missing values — empty strings, not NaN!
print(f'TotalCharges empty strings: {(df_raw["TotalCharges"] == " ").sum()}')


DATASET OVERVIEW

Shape: 7,043 rows x 21 columns

Target distribution:
Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn rate: 26.5%

--- Column Types ---
str        18
int64       2
float64     1
Name: count, dtype: int64

--- Missing Values ---
Explicit NaN count: 0
TotalCharges empty strings: 11


#### Observation: 
TotalCharges has empty strings masquerading as valid data. pandas read it as object (string) type instead of float.

#### Data Cleaning & Feature Engineering